# Lab 4 &middot; Finding edges in a noisy picture

**BITS F459 Computer Vision** &middot; 10 marks

Last session took a change in brightness, turned it into a number, and turned that
number into an edge map through the five steps of Canny. This lab runs those five steps
twice. First by hand on a 7&times;7 image small enough to see every pixel, generated from
your BITS ID. Then on a real photograph that has had sensor noise added to it.

| | |
|:--|--:|
| **Part A** &nbsp; all five steps, on your own 7&times;7 | 5 marks |
| **Part B** &nbsp; the same five steps on a real picture: noise and smoothing | 2 marks |
| **Part C** &nbsp; the two gradient thresholds | 3 marks |

**Three rules.**

1. Put your BITS ID in the first cell. Every number in this lab is generated from it, so
   nobody else has your figures.
2. Run the cells in order, top to bottom.
3. **The last cell must be the last thing you run.** It collects your answers. If you
   change anything above it afterwards, come back down and run it again before saving.

When you have finished: `File > Download > Download .ipynb`, rename the file to exactly
`lab04.ipynb`, and upload it to the root of your own repository.

---
## Setup

Type your BITS ID between the quotes, exactly as it appears on your ID card, then run
the cell.

In [ ]:
BITS_ID = ""          # for example "2024A7PS0123U"

In [ ]:
import base64, hashlib, io, json
from datetime import datetime, timezone

import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
from PIL import Image

assert BITS_ID.strip(), "Put your BITS ID in the cell above first."
SEED = hashlib.sha256(BITS_ID.strip().upper().encode()).hexdigest()
print("BITS ID :", BITS_ID.strip().upper())
print("lab seed:", SEED[:8])

### The five steps, as functions

The steps from the lecture, written out once so the rest of the lab can call them. Read
them, but do not change them.

| function | which step |
|:--|:--|
| `smooth(img, k)` | step 1, the Gaussian. `k = 0` means no smoothing at all |
| `gradient(img)` | step 2, returns Gx, Gy, the length and the angle |
| `round_direction(angle)` | the angle rounded to 0&deg;, 45&deg;, 90&deg; or 135&deg; |
| `thin(mag, angle)` | step 3, non-maximum suppression |
| `link(thinned, t_low, t_high)` | steps 4 and 5, the two thresholds and hysteresis |
| `find_edges(img, blur, t_low, t_high)` | all five, in order |
| `match_score(found, reference)` | how well one edge map matches another, 1.00 is perfect |

`smooth(img, k)` uses the kernel from the lecture. At `k = 3` it is exactly
(1/16) &times; [1 2 1; 2 4 2; 1 2 1], separated into [1 2 1] / 4 across and down. The
larger sizes are the same family continued: 5 is [1 4 6 4 1] / 16, and so on.

In [ ]:
import numpy as np
from scipy import ndimage

SX = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
SY = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float)


def binomial(k):
    """The 1-D smoothing kernel of length k. k=3 gives [1 2 1]/4."""
    row = np.array([1.0])
    for _ in range(k - 1):
        row = np.convolve(row, [1.0, 1.0])
    return row / row.sum()


def smooth(img, k):
    """Step 1. k = 0 means no smoothing at all."""
    if k in (0, 1):
        return np.asarray(img, float)
    r = binomial(k)
    a = ndimage.convolve1d(np.asarray(img, float), r, axis=1, mode="nearest")
    return ndimage.convolve1d(a, r, axis=0, mode="nearest")


def gradient(img):
    """Step 2. Gx, Gy, the length and the angle, exactly as on the slides."""
    a = np.asarray(img, float)
    # correlate, not convolve: the kernel is applied as it is written on the slide,
    # which is what a student doing it by hand will do. convolve() flips it and
    # returns the opposite sign for Gx and Gy.
    gx = ndimage.correlate(a, SX, mode="nearest") / 8.0
    gy = ndimage.correlate(a, SY, mode="nearest") / 8.0
    mag = np.hypot(gx, gy)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0
    return gx, gy, mag, ang


def round_direction(angle):
    """The angle rounded to one of the four lines that fit on a grid."""
    a = np.asarray(angle, float) % 180.0
    out = np.zeros(a.shape, int)
    out[(a >= 22.5) & (a < 67.5)] = 45
    out[(a >= 67.5) & (a < 112.5)] = 90
    out[(a >= 112.5) & (a < 157.5)] = 135
    return out


_STEP = {0: (0, 1), 45: (-1, 1), 90: (-1, 0), 135: (-1, -1)}


def thin(mag, angle):
    """Step 3. Keep a pixel only if it beats both neighbours across its own edge."""
    d = round_direction(angle)
    out = np.zeros_like(mag)
    p = np.pad(mag, 1)
    h, w = mag.shape
    for k, (dr, dc) in _STEP.items():
        a = p[1-dr:1-dr+h, 1-dc:1-dc+w]
        b = p[1+dr:1+dr+h, 1+dc:1+dc+w]
        keep = (d == k) & (mag > a) & (mag >= b)
        out[keep] = mag[keep]
    return out


def link(thinned, t_low, t_high):
    """Steps 4 and 5. Strong pixels, plus the weak ones that touch them."""
    strong = thinned >= t_high
    weak = (thinned >= t_low) & ~strong
    lab, n = ndimage.label(strong | weak, structure=np.ones((3, 3), int))
    if n == 0:
        return np.zeros(thinned.shape, np.uint8)
    good = np.zeros(n + 1, bool)
    good[np.unique(lab[strong])] = True
    good[0] = False
    return (good[lab]).astype(np.uint8) * 255


def find_edges(img, blur, t_low, t_high):
    """All five steps, in order."""
    s = smooth(img, blur)
    _, _, mag, ang = gradient(s)
    return link(thin(mag, ang), t_low, t_high)


def match_score(found, reference):
    """How well one edge map matches another. 1.00 is a perfect match."""
    a, b = found > 0, reference > 0
    both = np.logical_and(a, b).sum()
    if both == 0:
        return 0.0
    return float(2 * both / (a.sum() + b.sum()))

---
# Part A &middot; All five steps, on your own picture &nbsp;&nbsp;<small>5 marks</small>

Your 7&times;7 image is **two bright blocks with a dark channel between them**, and one
faint speck somewhere in the background. Their brightness, their size and the speck all
come from your BITS ID.

Two blocks rather than one, because a single block in a corner only ever produces
gradients pointing one way. With the channel between them, all four directions appear in
your picture and you will use every one of them.

The cell below runs steps 1 and 2 for you and prints five grids.

* **I**, your image.
* **S**, the image after the Gaussian, with **one value missing**.
* **|&nabla;f|**, the gradient length after step 2, with **one value missing**.
* **the angle**, and **the direction**, each with the same value missing.

The gaps are what you work out in A1.

In [ ]:
SPOTS = [(6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6),
         (5, 0), (5, 5), (5, 6), (4, 0), (4, 5), (4, 6), (3, 6)]
STEP = {0: (0, 1), 45: (-1, 1), 90: (-1, 0), 135: (-1, -1)}


def classify(th, t_low, t_high):
    # steps 4 and 5, split out so each class can be seen separately
    strong = th >= t_high
    weak = (th >= t_low) & ~strong
    lab, n = ndimage.label(strong | weak, structure=np.ones((3, 3), int))
    keep = np.zeros(n + 1, bool)
    keep[np.unique(lab[strong])] = True
    keep[0] = False
    return strong, weak, weak & keep[lab], weak & ~keep[lab]


def clean_pixels(mag, ang, spot):
    # one pixel per direction whose answer is not on the boundary of anything
    d = round_direction(ang)
    out = {0: [], 45: [], 90: [], 135: []}
    for rr in range(1, 6):
        for cc in range(1, 6):
            if abs(rr - spot[0]) <= 1 and abs(cc - spot[1]) <= 1:
                continue
            t = ang[rr, cc] % 180.0
            margin = min(abs(t - b) for b in (0., 22.5, 67.5, 112.5, 157.5, 180.))
            k = int(d[rr, cc])
            dr, dc = STEP[k]
            v, a1, a2 = mag[rr, cc], mag[rr - dr, cc - dc], mag[rr + dr, cc + dc]
            if v > .08 * mag.max() and margin >= 5. \
               and min(abs(v - a1), abs(v - a2)) >= .02 * mag.max():
                out[k].append((rr, cc))
    return out


def build_picture(seed_hex):
    r = np.random.default_rng(int(seed_hex[8:16], 16))
    hi = int(r.choice([160, 176, 192, 208, 224, 240]))
    lo = int(r.choice([16, 32, 48, 64, 80]))
    ac = int(r.choice([1, 2]))                                   # left block, columns 0..ac
    bc = int(r.choice([c for c in (ac + 3, ac + 4) if c <= 5]))  # right block, columns bc..6
    ar = int(r.choice([2, 3, 4]))                                # left block, rows 0..ar
    br = int(r.choice([x for x in (1, 2, 3) if x != ar]))        # right block, rows 0..br
    base = np.full((7, 7), float(lo))
    base[0:ar + 1, 0:ac + 1] = hi
    base[0:br + 1, bc:7] = hi

    r2 = np.random.default_rng(int(seed_hex[24:32], 16))
    for si in r2.permutation(len(SPOTS)):
        spot = SPOTS[int(si)]
        if base[spot] != lo:
            continue
        for frac in [x / 100 for x in range(30, 92, 3)]:
            img = base.copy()
            img[spot] = round(lo + frac * (hi - lo))
            S = smooth(img, 3)
            _, _, mag, ang = gradient(S)
            th = thin(mag, ang)
            m = th.max()
            tl, thi = round(.35 * m), round(.70 * m)
            _, _, kept, thrown = classify(th, tl, thi)
            if thrown.sum() != 1 or kept.sum() < 1:
                continue
            c = clean_pixels(mag, ang, spot)
            if not all(c[k] for k in (0, 45, 90, 135)):
                continue
            r3 = np.random.default_rng(int(seed_hex[16:24], 16))
            pix = {k: c[k][int(r3.integers(len(c[k])))] for k in (0, 45, 90, 135)}
            away = [(rr, cc) for rr in range(7) for cc in range(7)
                    if abs(S[rr, cc] - img[rr, cc]) > 0.5
                    and all(not (abs(rr - p[0]) <= 1 and abs(cc - p[1]) <= 1)
                            for p in pix.values())]
            if not away:
                continue
            return (img, hi, lo, spot, pix,
                    away[int(r3.integers(len(away)))], float(m))
    raise RuntimeError("no picture worked - tell your instructor")


IMG, HI, LO, SPOT, PIXELS, SMOOTH_AT, MAX_THIN = build_picture(SEED)
S7 = smooth(IMG, 3)
GX7, GY7, MAG7, ANG7 = gradient(S7)
THIN7 = thin(MAG7, ANG7)
# which of the four is blanked out and worked by hand in A1 also comes from your ID
A1_DIR = (0, 45, 90, 135)[int(np.random.default_rng(int(SEED[32:40], 16)).integers(4))]
PIXEL = PIXELS[A1_DIR]


def show_grid(title, arr, hide=None, dp=1, blank=None):
    print(title)
    for rr in range(arr.shape[0]):
        row = []
        for cc in range(arr.shape[1]):
            if hide is not None and (rr, cc) == tuple(hide):
                row.append("     ?")
            elif blank is not None and blank[rr, cc]:
                row.append("     .")
            else:
                row.append(f"{arr[rr, cc]:6.{dp}f}" if dp else f"{int(arr[rr, cc]):6d}")
        print(" ".join(row))
    print()


FLAT = MAG7 < 0.05           # no change in brightness there, so no direction to give

print(f"object {HI}, background {LO}, and a faint speck of {int(IMG[SPOT])} "
      f"at row {SPOT[0]}, column {SPOT[1]}\n")
show_grid("I, your image", IMG, dp=0)
show_grid("S, after step 1, the Gaussian", S7, hide=SMOOTH_AT)
show_grid("|grad f|, the gradient length after step 2", MAG7, hide=PIXEL)
show_grid("the angle, between 0 and 180 degrees", ANG7, hide=PIXEL, blank=FLAT)
show_grid("the direction, that angle rounded to 0, 45, 90 or 135",
          round_direction(ANG7), hide=PIXEL, dp=0, blank=FLAT)
print(f"the missing value of S is at row {SMOOTH_AT[0]}, column {SMOOTH_AT[1]}")
print(f"the missing gradient is at row {PIXEL[0]}, column {PIXEL[1]}")
print("a dot means the brightness does not change there at all\n")
print("your four pixels for A2, one for each direction:")
for _k in (0, 45, 90, 135):
    _mark = "   <- this is the one with the gap, from A1" if _k == A1_DIR else ""
    print(f"   {_k:3d} degrees -> row {PIXELS[_k][0]}, column {PIXELS[_k][1]}{_mark}")

### A1 &middot; steps 1 and 2 &nbsp;&nbsp;<small>1 mark</small>

**Step 1.** Work out the missing value of **S**. Take the 3&times;3 window of **I**
around that position and apply the Gaussian:

```
        1  2  1
 1/16 · 2  4  2
        1  2  1
```

Multiply each of the nine values by the weight sitting on top of it, add the nine
products, and divide the total by 16.

**If your position is on an edge of the image, read this first.** A 3&times;3 window
centred on the top row needs a row above it, and there is not one. The rule is that the
image is treated as if it continued by **repeating its nearest row or column outwards**.

So for a position in **column 0**, there is no column &minus;1, and the window uses a
copy of column 0 in its place:

```
 the image               the window you use
   a  b  ...               a │ a  b
   c  d  ...      -->       c │ c  d
   e  f  ...               e │ e  f
```

For a position in **row 0**, the row above is a copy of row 0. For a position in a
**corner**, both apply at once. If your position is not on an edge, nothing is repeated
and the nine values are simply the nine you can see.

**Step 2.** Now take the 3&times;3 window of **S** around the missing gradient and work
out:

```
        -1  0  1              -1 -2 -1
Sx  =   -2  0  2       Sy  =   0  0  0
        -1  0  1               1  2  1
```

* `Gx` &nbsp; the window with Sx laid over it, **divided by 8**
* `Gy` &nbsp; the window with Sy laid over it, **divided by 8**
* `length` &nbsp; &radic;(Gx&sup2; + Gy&sup2;)
* `angle` &nbsp; arctan(Gy / Gx), given **between 0 and 180 degrees**
* `direction` &nbsp; that angle rounded to 0, 45, 90 or 135

One decimal place on the first four.

In [ ]:
A1 = {
    "S_value":   None,      # the missing value of S
    "Gx":        None,
    "Gy":        None,
    "length":    None,
    "angle":     None,
    "direction": None,
}

assert all(v is not None for v in A1.values()), "Fill in all six."
assert A1["direction"] in (0, 45, 90, 135), "direction must be 0, 45, 90 or 135."

_truth = {"S_value": S7[SMOOTH_AT], "Gx": GX7[PIXEL], "Gy": GY7[PIXEL],
          "length": MAG7[PIXEL], "angle": ANG7[PIXEL]}
for _k, _v in _truth.items():
    _d = float(A1[_k]) - float(_v)
    print(f"  {_k:10s} {A1[_k]:>8}   " +
          ("correct" if abs(_d) <= 0.15 else ("too high" if _d > 0 else "too low")))
print(f"  {'direction':10s} {A1['direction']:>8}   " +
      ("correct" if A1["direction"] == int(round_direction(ANG7[PIXEL])) else "not this one"))

### A2 &middot; step 3, thinning, in all four directions &nbsp;&nbsp;<small>2 marks</small>

Step 3 compares a pixel with the two neighbours **across its own edge**: the two that lie
on the line of its direction.

| direction | the two neighbours |
|:--|:--|
| 0&deg; | left and right |
| 45&deg; | up-right and down-left |
| 90&deg; | above and below |
| 135&deg; | up-left and down-right |

The cell above named one pixel for each of the four directions. Do all four. For each,
read the two neighbours off the |&nabla;f| grid, give them in any order, and give the
verdict.

A pixel is kept only if it beats both of them. Where two values tie exactly, the
convention is a strict comparison on one side, so exactly one of the pair survives.

`verdict` is the word `"keep"` or the word `"zero"`.

> The 0&deg; pixel is the one whose gradient you worked out in A1, so you already have its
> value. The other three are printed for you.

In [ ]:
A2 = {
    0:   {"a": None, "b": None, "verdict": None},
    45:  {"a": None, "b": None, "verdict": None},
    90:  {"a": None, "b": None, "verdict": None},
    135: {"a": None, "b": None, "verdict": None},
}

for _k in (0, 45, 90, 135):
    _e = A2[_k]
    assert all(v is not None for v in _e.values()), f"Fill in all three for {_k} degrees."
    assert _e["verdict"] in ("keep", "zero"), 'verdict must be "keep" or "zero".'
    print(f"  {_k:3d} degrees, pixel ({PIXELS[_k][0]},{PIXELS[_k][1]}):  "
          f"{_e['a']} and {_e['b']}  ->  {_e['verdict']}")

### A3 &middot; step 4, your two thresholds &nbsp;&nbsp;<small>1 mark</small>

Run the cell. It prints your picture after step 3, and the **largest value left in it**.

The thresholds are set from that largest value, the same proportions used in the lecture,
where the largest gradient was 60 and the thresholds were 40 and 20:

```
 t_high  =  0.70 × the largest value,   rounded to a whole number
 t_low   =  0.35 × the largest value,   rounded to a whole number
```

Work out both. These are thresholds on |&nabla;f|. They are not brightness values and
they are not counts of anything.

In [ ]:
show_grid("after step 3, the thinned |grad f|", THIN7)
print(f"the largest value left after thinning is {MAX_THIN:.1f}")

In [ ]:
A3 = {
    "t_high": None,
    "t_low":  None,
}

assert all(v is not None for v in A3.values()), "Fill in both."
T_HIGH, T_LOW = int(A3["t_high"]), int(A3["t_low"])
print(f"  t_low {T_LOW}   t_high {T_HIGH}")
print("  " + ("these match the notebook" if
      (T_HIGH == round(0.70 * MAX_THIN) and T_LOW == round(0.35 * MAX_THIN))
      else "check the arithmetic - one of them is not what the rule gives"))

### A4 &middot; step 5, hysteresis &nbsp;&nbsp;<small>1 mark</small>

Using the thinned grid above and **your own two thresholds**:

* at or above `t_high` a pixel is **strong**
* between `t_low` and `t_high` it is **weak**
* below `t_low` it is thrown away at once

Step 5 then decides the weak ones. A weak pixel that touches a strong one, in any of the
eight directions, joins that contour and is kept. A weak pixel with no strong pixel
anywhere around it is noise and goes.

Your grid has **exactly one weak pixel that is thrown away**. Find it.

* `n_strong` &nbsp; how many pixels are strong
* `n_weak` &nbsp; how many are weak
* `n_discarded` &nbsp; how many are below `t_low` but were not already zero
* `thrown` &nbsp; `[row, column]` of the weak pixel that step 5 throws away
* `why` &nbsp; one sentence: why that one goes when the other weak pixels stay

In [ ]:
A4 = {
    "n_strong":    None,
    "n_weak":      None,
    "n_discarded": None,
    "thrown":      None,      # [row, column]
    "why":         None,      # one sentence
}

assert all(v is not None for v in A4.values()), "Fill in all five."
print("recorded:", {k: v for k, v in A4.items() if k != "why"})

---
# Part B &middot; The same five steps, on a real picture &nbsp;&nbsp;<small>2 marks</small>

The cell below loads the course banner and turns it to grey, because edges are about
brightness and nothing else. It builds the **reference**, the edge map of the clean
picture, which is the answer you are trying to get back. Then it adds sensor noise, with
a pattern generated from your BITS ID.

Everything from here is measured with `match_score`, which counts how much of your edge
map and the reference fall on the same pixels. 1.00 would be perfect. It punishes both
mistakes at once: edges you invented, and edges you missed.

In [ ]:
BANNER_B64 = "..."

In [ ]:
BANNER_PNG = base64.b64decode(BANNER_B64)
rgb = np.asarray(Image.open(io.BytesIO(BANNER_PNG)).convert("RGBA").convert("RGB"), float)
CLEAN = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]

REFERENCE = find_edges(CLEAN, 3, 10, 20)

_rng = np.random.default_rng(int(SEED[:8], 16))
NOISY = np.clip(CLEAN + _rng.normal(0, 20, CLEAN.shape), 0, 255)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.4))
for a, im, t in ((ax[0], CLEAN, "the picture, in grey"),
                 (ax[1], NOISY, "the same picture, with noise"),
                 (ax[2], 255 - REFERENCE, "the reference edge map")):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(t, fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

REF_PIXELS = int((REFERENCE > 0).sum())
print(f"the reference edge map has {REF_PIXELS:,} edge pixels, "
      f"{100 * (REFERENCE > 0).mean():.1f}% of the picture")

### B1 &middot; the smoothing curve, and what it says &nbsp;&nbsp;<small>1 mark</small>

Run the cell. It runs all five steps six times over, changing only the amount of
smoothing and holding the thresholds at 10 and 20 throughout.

Then fill in the box. The score is low at both ends of that table, and the reason is
different at each end: one end finds far too much, the other finds far too little.

* `scores` &nbsp; the six scores, to three decimal places
* `pixels_at_0` &nbsp; the edge pixel count with no smoothing at all
* `extra_pixels` &nbsp; how many **more** that is than the reference has
* `because` &nbsp; one sentence: why the score is low at the heavy-smoothing end, when
  there is hardly any noise left in the picture

In [ ]:
print(f"{'blur':>5} {'score':>8} {'edge pixels':>13}")
CURVE = {}
for _k in (0, 3, 5, 7, 9, 11):
    _e = find_edges(NOISY, _k, 10, 20)
    CURVE[_k] = (match_score(_e, REFERENCE), int((_e > 0).sum()))
    print(f"{_k:5d} {CURVE[_k][0]:8.3f} {CURVE[_k][1]:13,d}")

In [ ]:
B1 = {
    "scores":       {0: None, 3: None, 5: None, 7: None, 9: None, 11: None},
    "pixels_at_0":  None,
    "extra_pixels": None,
    "because":      None,      # one sentence
}

assert all(v is not None for v in B1["scores"].values()), "Fill in all six scores."
assert all(B1[k] is not None for k in ("pixels_at_0", "extra_pixels", "because"))
print("recorded:", B1["scores"], "| extra", B1["extra_pixels"])

### B2 &middot; choose the smoothing &nbsp;&nbsp;<small>1 mark</small>

Put the smoothing size you want into `SMOOTH`. The thresholds stay at 10 and 20.

The mark is for a score of **0.683** or better. Only one of the six sizes reaches it.

In [ ]:
SMOOTH = {
    "blur": 0,          # one of 0, 3, 5, 7, 9, 11
}

_e = find_edges(NOISY, SMOOTH["blur"], 10, 20)
B_SCORE = match_score(_e, REFERENCE)
print(f"  blur {SMOOTH['blur']}   score {B_SCORE:.3f}   edge pixels {int((_e > 0).sum()):,}")
print(f"  {'0.683 reached' if B_SCORE >= 0.683 else '0.683 not yet'}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].imshow(255 - _e, cmap="gray"); ax[0].set_title("yours", fontsize=10); ax[0].axis("off")
ax[1].imshow(255 - REFERENCE, cmap="gray"); ax[1].set_title("the reference", fontsize=10)
ax[1].axis("off")
plt.tight_layout(); plt.show()

---
# Part C &middot; The two gradient thresholds &nbsp;&nbsp;<small>3 marks</small>

Now all three settings are yours: the smoothing, and both gradient thresholds. These are
thresholds on |&nabla;f|, the same kind of number you worked out in A3.

There are two different things worth having, and they are not the same settings.

**C1, the highest score you can reach.** Your Part B answer is not the best there is:
holding the thresholds at 10 and 20 was a restriction, and lifting it buys more. The mark
is for beating your own Part B score by **0.010** or more.

**C2, the smallest edge map.** A high score is not the only thing worth having. Two edge
maps can score the same while one is a thin, joined-up outline and the other is that
outline plus a scattering of short broken fragments. The fragments cost nothing in score
and everything in usefulness. So: hold the score at **0.60** or better, and leave as few
edge pixels switched on as you can. The mark is for fewer than **13,000**.

The leaderboard ranks C2, the pixel count with the score still at 0.60 or above.

In [ ]:
BEST = {
    "blur":   0,        # one of 0, 3, 5, 7, 9, 11
    "t_low":  10,
    "t_high": 20,
}

_e = find_edges(NOISY, BEST["blur"], BEST["t_low"], BEST["t_high"])
BEST_SCORE = match_score(_e, REFERENCE)
print(f"  blur {BEST['blur']}   thresholds {BEST['t_low']} / {BEST['t_high']}")
print(f"  score {BEST_SCORE:.3f}      your Part B score was {B_SCORE:.3f}"
      f"      gain {BEST_SCORE - B_SCORE:+.3f}")
print(f"  {'0.010 gained' if BEST_SCORE - B_SCORE >= 0.010 else 'not 0.010 yet'}")

In [ ]:
FEWEST = {
    "blur":   0,        # one of 0, 3, 5, 7, 9, 11
    "t_low":  10,
    "t_high": 20,
}

_e = find_edges(NOISY, FEWEST["blur"], FEWEST["t_low"], FEWEST["t_high"])
C_SCORE, C_PIXELS = match_score(_e, REFERENCE), int((_e > 0).sum())
print(f"  blur {FEWEST['blur']}   thresholds {FEWEST['t_low']} / {FEWEST['t_high']}")
print(f"  score {C_SCORE:.3f}      edge pixels {C_PIXELS:,}")
print(f"  {'0.60 held' if C_SCORE >= 0.60 else 'score too low'}"
      f"      {'under 13,000' if C_SCORE >= 0.60 and C_PIXELS < 13000 else '13,000 not yet'}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].imshow(255 - _e, cmap="gray"); ax[0].set_title("yours", fontsize=10); ax[0].axis("off")
ax[1].imshow(255 - REFERENCE, cmap="gray"); ax[1].set_title("the reference", fontsize=10)
ax[1].axis("off")
plt.tight_layout(); plt.show()

### C3 &middot; switch step 5 off &nbsp;&nbsp;<small>1 mark</small>

In Part A you found one weak pixel that was thrown away because nothing strong was near
it, and others that survived because something was. This is the same thing on a
photograph with thousands of them.

Run your **C2 settings** again with `t_low` set equal to your `t_high`. That leaves no
weak class at all, so there is nothing for step 5 to rescue. Everything else stays the
same.

Report the score and the pixel count you get, and one sentence on what happened to the
contours. The settings are run again when this is marked, so the numbers have to be real.

In [ ]:
HYSTERESIS = {
    "score":   None,
    "pixels":  None,
    "what_happened": None,      # one sentence
}

assert all(v is not None for v in HYSTERESIS.values()), "Fill in all three."
_h = find_edges(NOISY, FEWEST["blur"], FEWEST["t_high"], FEWEST["t_high"])
print(f"  with hysteresis   score {C_SCORE:.3f}   edge pixels {C_PIXELS:,}")
print(f"  without it        score {match_score(_h, REFERENCE):.3f}   "
      f"edge pixels {int((_h > 0).sum()):,}")
print(f"  you reported      score {HYSTERESIS['score']}   "
      f"edge pixels {HYSTERESIS['pixels']:,}")

---
## Package your answers

This cell recomputes every measurement from the settings you have chosen, so it is
always current as of the moment you run it.

**Run it last.** If you go back and change anything above, come down here and run it
again before you save. Then `File > Download > Download .ipynb`, rename to exactly
`lab04.ipynb`, and upload it to your repository.

In [ ]:
_b = find_edges(NOISY, SMOOTH["blur"], 10, 20)
_best = find_edges(NOISY, BEST["blur"], BEST["t_low"], BEST["t_high"])
_c = find_edges(NOISY, FEWEST["blur"], FEWEST["t_low"], FEWEST["t_high"])
_h = find_edges(NOISY, FEWEST["blur"], FEWEST["t_high"], FEWEST["t_high"])

ANSWERS = {
    "lab": "lab04", "version": 3,
    "bits_id": BITS_ID.strip().upper(), "seed": SEED[:8],
    "generated": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "image": {"hi": int(HI), "lo": int(LO), "spot": [int(SPOT[0]), int(SPOT[1])],
              "max_thin": round(float(MAX_THIN), 2)},
    "pixels": {str(k): [int(v[0]), int(v[1])] for k, v in PIXELS.items()},
    "smooth_at": [int(SMOOTH_AT[0]), int(SMOOTH_AT[1])],
    "A1": {k: (int(v) if k == "direction" else float(v)) for k, v in A1.items()},
    "A2": {str(k): {"a": float(A2[k]["a"]), "b": float(A2[k]["b"]),
                    "verdict": str(A2[k]["verdict"]).strip().lower()}
           for k in (0, 45, 90, 135)},
    "A3": {"t_low": int(A3["t_low"]), "t_high": int(A3["t_high"])},
    "A4": {"n_strong": int(A4["n_strong"]), "n_weak": int(A4["n_weak"]),
           "n_discarded": int(A4["n_discarded"]),
           "thrown": [int(x) for x in A4["thrown"]], "why": str(A4["why"])},
    "B1": {"scores": {str(k): float(v) for k, v in B1["scores"].items()},
           "pixels_at_0": int(B1["pixels_at_0"]),
           "extra_pixels": int(B1["extra_pixels"]),
           "because": str(B1["because"])},
    "B2": {"blur": int(SMOOTH["blur"]),
           "score": round(float(match_score(_b, REFERENCE)), 4),
           "pixels": int((_b > 0).sum())},
    "C1": {"blur": int(BEST["blur"]), "t_low": int(BEST["t_low"]),
           "t_high": int(BEST["t_high"]),
           "score": round(float(match_score(_best, REFERENCE)), 4),
           "pixels": int((_best > 0).sum())},
    "C2": {"blur": int(FEWEST["blur"]), "t_low": int(FEWEST["t_low"]),
           "t_high": int(FEWEST["t_high"]),
           "score": round(float(match_score(_c, REFERENCE)), 4),
           "pixels": int((_c > 0).sum())},
    "C3": {"score": float(HYSTERESIS["score"]), "pixels": int(HYSTERESIS["pixels"]),
           "really": [round(float(match_score(_h, REFERENCE)), 4), int((_h > 0).sum())],
           "what_happened": str(HYSTERESIS["what_happened"])},
}

print("===== LAB04 ANSWER BLOCK v3 =====")
print(json.dumps(ANSWERS, separators=(",", ":"), default=float))
print("===== END LAB04 ANSWER BLOCK =====")
print("\nPacked. Now download this notebook as lab04.ipynb and upload it to your repo.")